# Practical Statistics for Data Scientists (Python)
# Chapter 7 — Unsupervised Learning

> **Author:** Satrio Arjuna Putra  
> **Reference:** *Practical Statistics for Data Scientists* — Bruce, Bruce & Gedeck (O'Reilly)

---

## 📋 Chapter Summary

Chapter 7 covers **unsupervised learning** — finding structure in data *without* labeled responses.

| Topic | Key Concepts |
|-------|-------------|
| **PCA** | Dimensionality reduction; principal components; explained variance |
| **K-Means Clustering** | Partition data into K clusters; elbow method |
| **Hierarchical Clustering** | Build tree of clusters (dendrogram); no need to specify K |
| **Model-Based Clustering** | Gaussian Mixture Models (GMM); soft assignments |
| **Scaling** | Critical for distance-based methods |
| **Categorical Data** | Gower's distance; one-hot encoding |

---


## 🔧 Setup & Imports

In [ ]:
%matplotlib inline

import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import squareform

DATA_BASE = "https://raw.githubusercontent.com/gedeck/practical-statistics-for-data-scientists/master/data/"
SP500_DATA    = DATA_BASE + "sp500_data.csv.gz"
SP500_SECTORS = DATA_BASE + "sp500_sectors.csv"
LOAN_DATA     = DATA_BASE + "loan_data.csv.gz"
HOUSE_TASKS   = DATA_BASE + "housetasks.csv"

np.random.seed(42)
print("Libraries loaded!")


---

## 1. Principal Component Analysis (PCA)

### 📚 Theory

**PCA** is a linear dimensionality reduction technique that finds orthogonal directions (principal components) of **maximum variance**.

**Algorithm:**
1. Center and scale the data
2. Compute the covariance matrix
3. Find eigenvectors (directions) and eigenvalues (variances)
4. Sort by eigenvalue (descending) → principal components

**Key terms:**
- **Loadings:** Correlation between original features and each PC
- **Scores:** Projection of data onto PCs
- **Explained Variance Ratio:** Proportion of total variance in each PC

$$\text{PC}_k = \mathbf{X} \cdot \mathbf{v}_k$$

> **Key Insight:** PCA reduces noise, removes collinearity, and enables visualisation of high-dimensional data. It is unsupervised — it doesn't use the response variable.


In [ ]:
# PCA on oil stocks (XOM, CVX)
sp500_px = pd.read_csv(SP500_DATA, index_col=0)
oil_px = sp500_px[['XOM', 'CVX']].dropna()
print("Oil stock data shape:", oil_px.shape)
print(oil_px.head())


In [ ]:
# Fit PCA to oil stocks
pca2 = PCA(n_components=2)
pca2.fit(oil_px)

loadings = pd.DataFrame(pca2.components_, columns=oil_px.columns,
                         index=['PC1', 'PC2'])
print("Loadings (direction each PC points):")
print(loadings)
print(f"
Explained Variance Ratio: {pca2.explained_variance_ratio_}")
print(f"PC1 explains {pca2.explained_variance_ratio_[0]*100:.1f}% of variance")


In [ ]:
# Visualise PCA on oil stocks
scores = pd.DataFrame(pca2.transform(oil_px), columns=['PC1', 'PC2'], index=oil_px.index)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Original space
axes[0].scatter(oil_px['XOM'], oil_px['CVX'], alpha=0.3, s=10)
# Draw PC directions
center = oil_px.mean()
for i, (pc, color) in enumerate(zip(pca2.components_, ['red', 'blue'])):
    scale = pca2.explained_variance_[i] ** 0.5
    axes[0].annotate('', xy=(center[0] + pc[0]*scale*3, center[1] + pc[1]*scale*3),
                    xytext=(center[0], center[1]),
                    arrowprops=dict(arrowstyle='->', color=color, lw=2))
axes[0].set_xlabel('XOM Returns')
axes[0].set_ylabel('CVX Returns')
axes[0].set_title('Oil Stocks — Original Space + PC Directions')

# PCA space
axes[1].scatter(scores['PC1'], scores['PC2'], alpha=0.3, s=10)
axes[1].axhline(0, color='k', linestyle='--', alpha=0.3)
axes[1].axvline(0, color='k', linestyle='--', alpha=0.3)
axes[1].set_xlabel('PC1 (Market Direction)')
axes[1].set_ylabel('PC2 (XOM vs CVX)')
axes[1].set_title('Oil Stocks — PCA Space')

plt.tight_layout()
plt.show()


In [ ]:
# PCA on S&P 500 — many stocks
sp500_sym = pd.read_csv(SP500_SECTORS)
top_symbols = sp500_sym['symbol'].tolist()[:20]  # use top 20
top_sp = sp500_px[top_symbols].dropna()

scaler = StandardScaler()
top_sp_scaled = scaler.fit_transform(top_sp)

sp_pca = PCA()
sp_pca.fit(top_sp_scaled)

# Scree plot
explained = sp_pca.explained_variance_ratio_
cumulative = np.cumsum(explained)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(1, len(explained[:10]) + 1), explained[:10] * 100)
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance (%)')
axes[0].set_title('Scree Plot')

axes[1].plot(range(1, len(cumulative[:15]) + 1), cumulative[:15] * 100, 'b-o')
axes[1].axhline(80, color='red', linestyle='--', label='80% threshold')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Explained Variance (%)')
axes[1].set_title('Cumulative Explained Variance')
axes[1].legend()

plt.tight_layout()
plt.show()
print(f"Components to explain 80% variance: {sum(cumulative < 0.8) + 1}")


---

## 2. K-Means Clustering

### 📚 Theory

**K-Means** partitions data into K clusters by minimising within-cluster variance (inertia):

$$\text{Inertia} = \sum_{k=1}^{K} \sum_{x \in C_k} \|x - \mu_k\|^2$$

**Algorithm (Lloyd's):**
1. Randomly initialise K centroids
2. Assign each point to nearest centroid
3. Update centroids as cluster means
4. Repeat 2-3 until convergence

**Choosing K — Elbow Method:** Plot inertia vs K; pick the "elbow" where improvement slows.

**Sensitivity:** K-Means is sensitive to initialisation and outliers; use `n_init` > 1.

> **Key Insight:** K-Means assumes clusters are spherical and of similar size. It requires K to be specified in advance. For clusters of irregular shape, use DBSCAN or GMM.


In [ ]:
# K-Means on oil stocks
df_oil = sp500_px.loc[sp500_px.index >= '2011-01-01', ['XOM', 'CVX']].dropna()

kmeans4 = KMeans(n_clusters=4, n_init=10, random_state=42)
kmeans4.fit(df_oil)

centers = pd.DataFrame(kmeans4.cluster_centers_, columns=['XOM', 'CVX'])
print("Cluster Centers:")
print(centers)

df_plot = df_oil.copy()
df_plot['cluster'] = kmeans4.labels_.astype(str)

fig, ax = plt.subplots(figsize=(7, 5))
colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3']
for i, (cluster, grp) in enumerate(df_plot.groupby('cluster')):
    ax.scatter(grp['XOM'], grp['CVX'], c=colors[i], alpha=0.5, s=20, label=f'Cluster {cluster}')
ax.scatter(centers['XOM'], centers['CVX'], c='black', marker='X', s=150, zorder=5, label='Centroids')
ax.set_xlabel('XOM Daily Return')
ax.set_ylabel('CVX Daily Return')
ax.set_title('K-Means Clustering: Oil Stocks (K=4)')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Elbow Method: choosing optimal K
inertias = []
k_range = range(2, 15)
for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    km.fit(df_oil)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(k_range, inertias, 'b-o')
plt.xlabel('Number of Clusters K')
plt.ylabel('Inertia (Within-Cluster Sum of Squares)')
plt.title('Elbow Method for Optimal K')
plt.tight_layout()
plt.show()


---

## 3. Hierarchical Clustering

### 📚 Theory

**Hierarchical clustering** builds a **dendrogram** (tree of clusters) by successively merging (agglomerative) or splitting (divisive) clusters.

**Linkage methods (how to measure cluster distances):**

| Method | Distance | Tendency |
|--------|----------|----------|
| **Single** | Min distance | Elongated clusters; chaining |
| **Complete** | Max distance | Compact clusters |
| **Average** | Mean distance | Compromise |
| **Ward** | Minimise variance increase | Round clusters; most popular |

**Advantages over K-Means:**
- No need to specify K in advance
- Dendrogram visualises the merging structure
- Deterministic (no random initialisation)

> **Key Insight:** Cut the dendrogram at a height to get K clusters. Ward's linkage tends to produce compact, equal-sized clusters — usually the best default choice.


In [ ]:
# Hierarchical clustering on selected stocks
syms_hc = ['AAPL', 'AMZN', 'AXP', 'COP', 'CSCO', 'CVX',
           'GOOGL', 'JPM', 'MSFT', 'SLB', 'TGT', 'USB', 'WFC', 'WMT', 'XOM']
syms_avail = [s for s in syms_hc if s in sp500_px.columns]
df_hc = sp500_px.loc[sp500_px.index >= '2011-01-01', syms_avail].dropna()

# Compute distance matrix (1 - correlation)
corr_matrix = df_hc.corr()
dist_matrix = np.sqrt(0.5 * (1 - corr_matrix))

# Linkage
Z = linkage(squareform(dist_matrix.values), method='ward')

fig, ax = plt.subplots(figsize=(10, 5))
dendrogram(Z, labels=syms_avail, ax=ax, color_threshold=0.5)
plt.xticks(rotation=45)
plt.ylabel('Distance')
plt.title('Hierarchical Clustering Dendrogram — Selected S&P 500 Stocks')
plt.tight_layout()
plt.show()


In [ ]:
# Cut dendrogram at K=4 clusters
cluster_labels = fcluster(Z, t=4, criterion='maxclust')
cluster_df = pd.Series(cluster_labels, index=syms_avail, name='Cluster')
for cluster, stocks in cluster_df.groupby(cluster_df):
    print(f"Cluster {cluster}: {', '.join(stocks.index.tolist())}")


---

## 4. Gaussian Mixture Models (GMM)

### 📚 Theory

**GMM** is a probabilistic, model-based clustering approach. It assumes data is generated from a **mixture of Gaussian distributions**:

$$p(x) = \sum_{k=1}^{K} \pi_k \cdot \mathcal{N}(x \mid \mu_k, \Sigma_k)$$

**Key differences from K-Means:**
- **Soft assignments:** Each point has a probability of belonging to each cluster
- **Flexible shapes:** Clusters can be elliptical (not just spherical)
- Fitted via the **EM (Expectation-Maximisation) algorithm**

**Model selection:** Use **BIC** or **AIC** to choose K.

> **Key Insight:** GMM is more flexible than K-Means and provides probabilistic cluster membership. Use BIC (lower = better) to select the number of components.


In [ ]:
# Gaussian Mixture Model on oil stocks
gmm = GaussianMixture(n_components=2, random_state=42)
gmm.fit(df_oil)

df_gmm = df_oil.copy()
df_gmm['cluster'] = gmm.predict(df_oil).astype(str)
df_gmm['prob_cluster0'] = gmm.predict_proba(df_oil)[:, 0]

print("GMM Means:
", pd.DataFrame(gmm.means_, columns=['XOM', 'CVX']))
print("
GMM Covariances:", gmm.covariances_.shape)

fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(df_gmm['XOM'], df_gmm['CVX'],
                     c=df_gmm['prob_cluster0'], cmap='RdBu', alpha=0.5, s=15)
plt.colorbar(scatter, ax=ax, label='Prob(Cluster 0)')
ax.set_xlabel('XOM Daily Return')
ax.set_ylabel('CVX Daily Return')
ax.set_title('Gaussian Mixture Model: Oil Stocks (K=2, soft assignments)')
plt.tight_layout()
plt.show()


In [ ]:
# Select optimal K for GMM using BIC
bic_scores = []
aic_scores = []
k_range_gmm = range(1, 9)
for k in k_range_gmm:
    gmm_k = GaussianMixture(n_components=k, random_state=42)
    gmm_k.fit(df_oil)
    bic_scores.append(gmm_k.bic(df_oil))
    aic_scores.append(gmm_k.aic(df_oil))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(k_range_gmm, bic_scores, 'b-o', label='BIC')
ax.plot(k_range_gmm, aic_scores, 'r-s', label='AIC')
ax.axvline(np.argmin(bic_scores) + 1, color='blue', linestyle='--',
           label=f'Best BIC K={np.argmin(bic_scores)+1}')
ax.set_xlabel('Number of Components K')
ax.set_ylabel('Score (lower = better)')
ax.set_title('GMM Model Selection: BIC and AIC')
ax.legend()
plt.tight_layout()
plt.show()


---

## 5. Scaling and Preprocessing for Clustering

### 📚 Theory

Distance-based algorithms (K-Means, KNN, Hierarchical) are **sensitive to feature scale**.

**Feature Scaling methods:**

| Method | Formula | Use When |
|--------|---------|---------|
| **Standardisation (Z-score)** | $(x - \mu) / \sigma$ | Features are approximately normal |
| **Min-Max** | $(x - \min) / (\max - \min)$ | Bounded range needed |
| **Robust Scaling** | $(x - \text{median}) / \text{IQR}$ | Data has outliers |

**Categorical variables:**
- One-hot encode and standardise (or use Gower's distance)
- **Gower's distance** handles mixed data types natively

> **Key Insight:** Always scale numeric features before applying distance-based clustering. Failing to scale means high-magnitude features dominate the distance calculation.


In [ ]:
# Effect of scaling on K-Means clustering
loan_data = pd.read_csv(LOAN_DATA)
loan_data = loan_data.dropna(subset=['dti', 'payment_inc_ratio', 'annual_inc'])

# Unscaled
X_unscaled = loan_data[['annual_inc', 'dti']].head(1000)
km_unscaled = KMeans(n_clusters=3, n_init=10, random_state=42)
km_unscaled.fit(X_unscaled)

# Scaled
scaler_loan = StandardScaler()
X_scaled_loan = scaler_loan.fit_transform(X_unscaled)
km_scaled = KMeans(n_clusters=3, n_init=10, random_state=42)
km_scaled.fit(X_scaled_loan)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, km, X, title in [
    (axes[0], km_unscaled, X_unscaled.values, 'Unscaled'),
    (axes[1], km_scaled, X_scaled_loan, 'Scaled (StandardScaler)'),
]:
    ax.scatter(X[:, 0], X[:, 1], c=km.labels_, cmap='tab10', s=10, alpha=0.5)
    ax.set_xlabel('Annual Income' if 'Un' in title else 'Annual Income (scaled)')
    ax.set_ylabel('DTI' if 'Un' in title else 'DTI (scaled)')
    ax.set_title(f'K-Means Clustering ({title})')
plt.tight_layout()
plt.show()


In [ ]:
# One-hot encoding for categorical variables then cluster
loan_cat = loan_data[['dti', 'payment_inc_ratio', 'home_', 'purpose_']].head(500).dropna()
loan_encoded = pd.get_dummies(loan_cat, drop_first=True)

scaler_cat = StandardScaler()
loan_encoded_scaled = scaler_cat.fit_transform(loan_encoded)

km_cat = KMeans(n_clusters=4, n_init=10, random_state=42)
km_cat.fit(loan_encoded_scaled)

# Show cluster composition
loan_cat_result = loan_cat.copy()
loan_cat_result['cluster'] = km_cat.labels_
print("Cluster size:")
print(loan_cat_result['cluster'].value_counts().sort_index())
print("
Mean DTI by cluster:")
print(loan_cat_result.groupby('cluster')['dti'].mean().round(2))


---

## ✅ Chapter 7 Key Takeaways

1. **PCA:** Reduces dimensionality while preserving maximum variance; use scree plot to choose number of components.
2. **K-Means:** Fast and simple; use elbow method to choose K; sensitive to outliers and scale.
3. **Hierarchical Clustering:** No need to specify K; dendrogram shows cluster structure; Ward's linkage is usually best.
4. **GMM:** Soft, probabilistic cluster assignments; flexible elliptical shapes; use BIC to select K.
5. **Scaling is critical:** Always standardise numeric features before distance-based clustering.
6. **Categorical data:** Use one-hot encoding + scaling, or Gower's distance for mixed data.
7. **Unsupervised ≠ Validation-free:** Always evaluate clusters qualitatively and with domain knowledge.

---
*Reference: Chapter 7, Practical Statistics for Data Scientists (O'Reilly)*
